# BarbellOS: próba animacji ruchu z dwóch klatek (Wan, model wideo)

Cel: sprawdzić, czy otwarty model wideo zrobi płynny przysiad z dwóch ilustracji (góra i dół), zanim w to wejdziemy.

**Zanim uruchomisz:** menu **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → A100 GPU** (jeśli jest; inaczej L4) i włącz **Wysoka pamięć RAM**. Potem **Środowisko wykonawcze → Uruchom wszystko**.

Pierwsze uruchomienie pobiera model (kilkadziesiąt GB), więc potrwa 15–30 minut. Generowanie: kilka do kilkunastu minut. Na końcu zobaczysz film pod ostatnią komórką i przycisk pobrania.

## 1. Karta graficzna

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.0f} GB")

## 2. Instalacja (diffusers z gałęzi głównej, jak każe karta modelu)

In [ ]:
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate ftfy imageio imageio-ffmpeg

## 3. Klatki przysiadu z repozytorium BarbellOS

Te same klatki, które aplikacja przełącza teraz (`web/public/technique`). Repo jest publiczne, więc nic nie trzeba wgrywać.

In [ ]:
from diffusers.utils import load_image

BASE = "https://raw.githubusercontent.com/Preeksik/BarbellOS/main/web/public/technique"
EXERCISE = "przysiad"
first_frame = load_image(f"{BASE}/{EXERCISE}-1.webp").convert("RGB")
last_frame = load_image(f"{BASE}/{EXERCISE}-2.webp").convert("RGB")
print(first_frame.size, last_frame.size)
display(first_frame.resize((first_frame.width // 3, first_frame.height // 3)))
display(last_frame.resize((last_frame.width // 3, last_frame.height // 3)))

## 4. Model: Wan 2.1 FLF2V 14B (klatka startowa + końcowa)

Jedyny model Wan z udokumentowanym trybem „pierwsza i ostatnia klatka” w `diffusers`. Części ładujemy na procesor i przenosimy na kartę tylko na czas liczenia (`enable_model_cpu_offload`), żeby zmieścić się w pamięci karty.

In [ ]:
import torch
from transformers import CLIPVisionModel
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline

model_id = "Wan-AI/Wan2.1-FLF2V-14B-720P-diffusers"
image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", torch_dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(
    model_id, vae=vae, image_encoder=image_encoder, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
print("model gotowy")

## 5. Generowanie ruchu góra → dół

Rozdzielczość ok. 480p (szybciej), 49 klatek (ok. 3 s przy 16 kl./s). Potem klip odtwarzamy wstecz, więc pętla wychodzi góra → dół → góra bez szwu.

In [ ]:
import math
import numpy as np

def fit(image, max_area):
    # Wymiary podzielne przez krok modelu, z zachowaniem proporcji klatki.
    mod = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
    ratio = image.height / image.width
    height = round(math.sqrt(max_area * ratio)) // mod * mod
    width = round(math.sqrt(max_area / ratio)) // mod * mod
    return image.resize((width, height)), height, width

first, height, width = fit(first_frame, 480 * 832)
last, _, _ = fit(last_frame, 480 * 832)
print("rozdzielczość", width, "x", height)

prompt = (
    "Anatomical muscle illustration of a man doing a barbell back squat in a squat rack, side view. "
    "He squats down smoothly and under control. The barbell stays rigid, straight and level on his upper back, "
    "the plates do not change size, his hands stay on the bar. The rack and the floor stay completely still. "
    "Pure white background, studio light, smooth realistic motion."
)
negative_prompt = (
    "bent barbell, warped plates, extra limbs, extra fingers, hands passing through the bar, morphing, "
    "flicker, camera movement, zoom, text, watermark, blurry, low quality"
)
generator = torch.Generator(device="cuda").manual_seed(7)
frames = pipe(
    image=first, last_image=last, prompt=prompt, negative_prompt=negative_prompt,
    height=height, width=width, num_frames=49, num_inference_steps=30, guidance_scale=5.5,
    generator=generator,
).frames[0]
print("klatek:", len(frames))

## 6. Pętla, podgląd i pobranie

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

loop = list(frames) + list(frames[::-1])[1:-1]
path = f"/content/{EXERCISE}-wan.mp4"
export_to_video(loop, path, fps=16)
data = b64encode(open(path, "rb").read()).decode()
display(HTML(f'<video src="data:video/mp4;base64,{data}" autoplay loop muted playsinline width="360"></video>'))
files.download(path)

## Co sprawdzić

- Czy **gryf jest prosty** i talerze mają stały rozmiar przez cały ruch.
- Czy **dłonie zostają na gryfie**.
- Czy ruch w połowie wygląda jak przysiad (biodra w tył i w dół), a nie dziwne przenikanie.
- Czy stojak i podłoga stoją w miejscu.

Wynik (plik `przysiad-wan.mp4`) wrzuć do `exports/technika/` i napisz agentowi.